# Stage 9 - render every figure for the write-up

`adversec/report.py` holds the plotting code but nothing called it, so the figures existed
only as functions. This notebook renders all of them, for both datasets, displays them
inline, and writes PNG + PDF into `figures/` ready to drop into the dissertation.

**Reads only `results/*.json`** - no models, no raw data, no GPU. It runs in seconds on any
machine with matplotlib, including a laptop with no torch installed, so figures can be
regenerated without re-running any experiment.

| Figure | What it shows | Source |
|---|---|---|
| `diversity_gradient` | unique attack signatures per class (log scale) | nb04 |
| `baseline_confusion` | RF vs CNN confusion matrices on clean data | nb03 |
| `perclass_robustness` | per-class F1 vs PGD epsilon, cross-validated | nb04 |
| `distance_mechanism` | distance-to-benign vs robustness (sign differs by dataset) | nb04 |
| `threat_sizing` | raw vs integer-rounded F1 and envelope rejection % | nb06 |
| `envelope_control` | envelope on CLEAN frames + standalone detector score | nb06 |
| `defence` | clean / transfer / white-box for baseline, static AT, Madry AT, RF | nb05 |
| `attack_grid` | every attack x model cell, incl. black-box | nb07 |
| `accuracy_trap` | leaky vs honest CV, with the inflation annotated | nb08 |


In [ ]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import matplotlib.pyplot as plt
from adversec import config, report

FIG_DIR = config.PROJECT_ROOT / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATASETS = ['ciciov2024', 'road']
DPI = 200            # print-quality raster; the PDF is vector and is what to use in Word/LaTeX
SAVE_PDF = True

PLOTS = [
    ('diversity_gradient',  report.plot_diversity_gradient),
    ('baseline_confusion',  report.plot_baseline_confusion),
    ('perclass_robustness', report.plot_perclass_robustness),
    ('distance_mechanism',  report.plot_distance_mechanism),
    ('threat_sizing',       report.plot_threat_sizing),
    ('envelope_control',    report.plot_envelope_control),
    ('defence',             report.plot_defence),
    ('attack_grid',         report.plot_attack_grid),
    ('accuracy_trap',       report.plot_accuracy_trap),
]
print('figures ->', FIG_DIR)

## Render, display and save

In [ ]:
written, missing = [], []
for name in DATASETS:
    for stem, fn in PLOTS:
        fig = fn(name)
        if fig is None:
            missing.append(f'{name}_{stem}')
            continue
        png = FIG_DIR / f'{name}_{stem}.png'
        fig.savefig(png, dpi=DPI, bbox_inches='tight')
        written.append(png.name)
        if SAVE_PDF:
            pdf = FIG_DIR / f'{name}_{stem}.pdf'
            fig.savefig(pdf, bbox_inches='tight')
            written.append(pdf.name)
        plt.show()          # display inline as well as save
        plt.close(fig)

print(f'\nwrote {len(written)} files to {FIG_DIR}')
if missing:
    print('MISSING (results artifact absent -- run the notebook that produces it):')
    for m in missing:
        print('   ', m)
else:
    print('every figure rendered for both datasets')

## Index of what was written

A quick manifest so the write-up can reference figures by filename without guessing.


In [ ]:
for f in sorted(FIG_DIR.glob('*.png')):
    print(f'  {f.name:44s} {f.stat().st_size / 1024:7.1f} KB')